**Table 5** (Tirado-Martin et al., 2024, *NeuroVoz*, Scientific Data) — Voice measurements extracted from the /a/ sustained vowel recordings. These are the columns in `audio_features.csv`.

| Parameter | Abbreviation | Unit Measure |
|---|---|---|
| **Perturbation Measures** | | |
| Absolute Jitter | Jitter | μSeconds |
| Relative Jitter | rJitter | % |
| Relative Average Perturbation | RAP | % |
| Pitch Period Perturbation Quotient | rPPQ | % |
| Smoothed Pitch Period Perturbation Quotient | rSPPQ | % |
| Absolute Shimmer | ShimmerDb | dB |
| Relative Shimmer | rShimmer | % |
| Amplitude Perturbation Quotient | APQ | % |
| Smoothed Amplitude Perturbation Quotient | sAPQ | % |
| Cepstral Peak Prominence | CPP | dB |
| **Noise Parameters** | | |
| Harmonics-to-Noise Ratio | HNR | dB |
| Cepstrum Harmonics-to-Noise Ratio | CHNR | dB |
| Glottal to Noise Excitation Ratio | GNE | Ratio |
| Normalised Noise Energy | NNE | dB |
| **Tremor Parameters** | | |
| Frequency Tremor Intensity Index | FTRI | Arbitrary Units |
| Amplitude Tremor Intensity Index | ATRI | Arbitrary Units |
| Fundamental Frequency Tremor Frequency | FFTR | Hz |
| Amplitude Tremor Frequency | ATRF | Hz |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

df = pd.read_csv("/Users/evelyn/Documents/claudecode/speechlab/Parkinsons/Spanish neurovoz_v3/audio_features/audio_features.csv")

def get_label(path):
    return 1 if "/PD_" in path else 0

df["label"] = df["AudioPath"].apply(get_label)
df.head()

In [ ]:
print(f"Shape: (rows = {df.shape[0]} audio files, cols = {df.shape[1]} features [18 MDVP measures + AudioPath + label])")
print("\nClass distribution:")
print(df["label"].value_counts())
print("\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df = df.dropna().reset_index(drop=True)
print(f"\nAfter dropping rows with missing values: {df.shape[0]} rows remain")

In [ ]:
features_to_plot = {
    "rJitter": "relative jitter",
    "rShimmer": "relative shimmer",
    "HNR": "harmonics to noise ratio",
    "GNE": "glottal to noise excitation",
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, (feature, description) in enumerate(features_to_plot.items()):
    sns.histplot(data=df, x=feature, hue="label", ax=axes[i], kde=True)
    axes[i].set_title(description)
    axes[i].legend(["HC (0)", "PD (1)"])

plt.tight_layout()
plt.show()

In [ ]:
feature_cols = [c for c in df.columns if c not in ("AudioPath", "label")]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "label": df["label"].map({0: "HC", 1: "PD"}),
})

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=pca_df, x="PC1", y="PC2",
    hue="label", alpha=0.4, s=15,
    palette={"HC": "steelblue", "PD": "tomato"},
    ax=ax,
)
ax.set_title(
    f"PCA of MDVP features (sustained vowels)\n"
    f"PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% variance, "
    f"PC2 explains {pca.explained_variance_ratio_[1]*100:.1f}%"
)
plt.tight_layout()
plt.show()

In [ ]:
X = df[feature_cols].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print(classification_report(y_test, rf.predict(X_test), target_names=["HC", "PD"]))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, rf.predict(X_test),
    display_labels=["HC", "PD"],
    ax=ax,
)
ax.set_title("Confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
# human-readable names, per Table 5 (NeuroVoz paper)
readable = {
    "Jitter": "Absolute Jitter (μs)",
    "rJitter": "Relative Jitter (%)",
    "RAP": "Relative Average Perturbation (%)",
    "rPPQ": "Pitch Period Perturbation Quotient (%)",
    "rSPPQ": "Smoothed Pitch Period Perturbation Quotient (%)",
    "ShimmerDb": "Absolute Shimmer (dB)",
    "rShimmer": "Relative Shimmer (%)",
    "APQ": "Amplitude Perturbation Quotient (%)",
    "sAPQ": "Smoothed Amplitude Perturbation Quotient (%)",
    "CPP": "Cepstral Peak Prominence (dB)",
    "HNR": "Harmonics-to-Noise Ratio (dB)",
    "CHNR": "Cepstrum Harmonics-to-Noise Ratio (dB)",
    "GNE": "Glottal to Noise Excitation Ratio",
    "NNE": "Normalised Noise Energy (dB)",
    "FTRI": "Frequency Tremor Intensity Index",
    "ATRI": "Amplitude Tremor Intensity Index",
    "FFTR": "Fundamental Frequency Tremor Frequency (Hz)",
    "ATRF": "Amplitude Tremor Frequency (Hz)",
}

importances = pd.Series(rf.feature_importances_, index=feature_cols)
top_readable = importances.sort_values().copy()
top_readable.index = [readable.get(f, f) for f in top_readable.index]

fig, ax = plt.subplots(figsize=(9, 7))
top_readable.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Feature importances (Random Forest) — all 18 MDVP features")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()